<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula05_%C3%A1rvore_de_decis%C3%A3o_atributos_discretos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ucimlrepo -q

In [6]:
from ucimlrepo import fetch_ucirepo

car_evaluation = fetch_ucirepo(id=19)

X = car_evaluation.data.features.to_numpy()
y = car_evaluation.data.targets.to_numpy()[:,0]

print(car_evaluation.variables)

       name     role         type demographic  \
0    buying  Feature  Categorical        None   
1     maint  Feature  Categorical        None   
2     doors  Feature  Categorical        None   
3   persons  Feature  Categorical        None   
4  lug_boot  Feature  Categorical        None   
5    safety  Feature  Categorical        None   
6     class   Target  Categorical        None   

                                         description units missing_values  
0                                       buying price  None             no  
1                           price of the maintenance  None             no  
2                                    number of doors  None             no  
3              capacity in terms of persons to carry  None             no  
4                           the size of luggage boot  None             no  
5                        estimated safety of the car  None             no  
6  evaulation level (unacceptable, acceptable, go...  None             no  

In [8]:
for i in range(X.shape[1]):
  values = set(X[:,i])
  print(f"{i}. {car_evaluation.variables['name'][i]}:\t{values}")

0. buying:	{'high', 'vhigh', 'med', 'low'}
1. maint:	{'high', 'vhigh', 'med', 'low'}
2. doors:	{'5more', '3', '4', '2'}
3. persons:	{'4', '2', 'more'}
4. lug_boot:	{'big', 'small', 'med'}
5. safety:	{'high', 'low', 'med'}


In [9]:
set(y)

{'acc', 'good', 'unacc', 'vgood'}

In [18]:
for label in set(y):
  print(f"{label}:\t{100*sum(y==label)/len(y):.4}%")

vgood:	3.762%
good:	3.993%
unacc:	70.02%
acc:	22.22%


In [77]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

def most_common(y):
  return max(set(y), key=lambda label: sum(y==label))

class ZeroR(BaseEstimator, ClassifierMixin):
  def fit(self, X, y):
    self.answer = most_common(y)
  def predict(self, X):
    return [self.answer]*X.shape[0]

model = ZeroR()
model.fit(X, y)
y_pred = model.predict(X)
print(accuracy_score(y, y_pred))

0.7002314814814815


In [78]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    stratify=y,
                                                    shuffle=True,
                                                    random_state=42)
model = ZeroR()
model.fit(X_train, y_train)
yhat = model.predict(X_test)
accuracy = accuracy_score(y_test, yhat)
print('Accuracy: %.3f' % (accuracy*100))

Accuracy: 69.942


In [138]:
import numpy as np

# características e valores aleatórios
class DecisionTree(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.feature = np.random.randint(X.shape[1])
        self.value = np.random.choice(list(set(X[:, self.feature])))
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0:
            self.equals_tree = DecisionTree().fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree().fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree()
model.fit(X, y)
ypred = model.predict(X)
print(accuracy_score(y, ypred))

0.7563657407407407


In [140]:
def gini(y):
  labels = list(set(y))
  x = 0
  for label in labels:
    label_prob = np.mean(y==label)
    x += label_prob**2
  return 1-x

print(gini(y_train))

0.4570454112310227


In [144]:
print(gini(np.ones(100)))

0.0


In [146]:
print(gini(np.arange(100)))

0.99


In [151]:
def impurity_value(x, y, value, impurity_function):
  equals = x == value
  equals_impurity = impurity_function(y[equals])
  not_equals_impurity = impurity_function(y[~equals])
  return (np.mean(equals)) * equals_impurity + (np.mean(~equals)) * not_equals_impurity

print(impurity_value(X_train[:,0], y_train, "vhigh", gini))

0.4482135562918824


In [162]:
def best_split(x, y, impurity_function):
  best_value = None
  best_value_impurity = float('inf')
  for value in set(x):
    value_impurity = impurity_value(x, y, value, impurity_function)
    if value_impurity < best_value_impurity:
      best_value = value
      best_value_impurity = value_impurity
  return best_value, best_value_impurity

print(best_split(X_train[:,2], y_train, gini))

('5more', np.float64(0.45634557119913965))


In [166]:
def best_feature(X, y, impurity_function):
  best_feature = None
  best_value = None
  best_value_impurity = float('inf')
  for feature in range(X.shape[1]):
    value, _ = best_split(X[:,feature], y, impurity_function)
    feature_impurity = impurity_value(X[:,feature], y, value, impurity_function)
    if feature_impurity < best_value_impurity:
      best_feature = feature
      best_value_impurity = feature_impurity
      best_value = value
  return best_feature, best_value, best_value_impurity

print(best_feature(X_train, y_train, gini))

(5, 'low', np.float64(0.38499511557696947))


In [167]:
class DecisionTree(BaseEstimator, ClassifierMixin):
    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0:
            self.equals_tree = DecisionTree().fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree().fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree()
model.fit(X, y)
ypred = model.predict(X)
print(accuracy_score(y, ypred))

1.0


In [168]:
model = DecisionTree()
model.fit(X_train, y_train)
ypred = model.predict(X_test)
print(accuracy_score(y_test, ypred))

0.9682080924855492


In [172]:
from sklearn.model_selection import cross_val_score, KFold

scores = cross_val_score(DecisionTree(), X, y, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.97687861 0.98554913 0.96242775 0.96811594 0.96811594]
0.9722174750774901


In [182]:
class DecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=9999999):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > 0 and sum(~equals) > 0 and self.max_depth>0:
            self.equals_tree = DecisionTree(self.max_depth-1).fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree(self.max_depth-1).fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree(5)
scores = cross_val_score(model, X, y, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.86127168 0.83815029 0.88150289 0.84057971 0.91014493]
0.8663298986344976


In [192]:
class DecisionTree(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=9999999, min_sample_split=2):
        self.max_depth = max_depth
        self.min_sample_split = min_sample_split

    def fit(self, X, y):
        self.feature, self.value, self.impurity = best_feature(X, y, gini)
        equals = X[:, self.feature] == self.value
        if sum(equals) > self.min_sample_split and \
           sum(~equals) > self.min_sample_split and \
           self.max_depth>0:
            self.equals_tree = DecisionTree(self.max_depth-1).fit(X[equals], y[equals])
            self.not_equals_tree = DecisionTree(self.max_depth-1).fit(X[~equals], y[~equals])
        else:
            self.answer = most_common(y)
        return self

    def predict(self, X):
        if hasattr(self, 'answer'):
            return [self.answer]*X.shape[0]
        else:
            equals = X[:, self.feature] == self.value
            return np.where(equals, self.equals_tree.predict(X), self.not_equals_tree.predict(X))

model = DecisionTree(20, 10)
scores = cross_val_score(model, X, y, cv=KFold(n_splits=5, shuffle=True))
print(scores)
print(np.mean(scores))

[0.9566474  0.95375723 0.94219653 0.97101449 0.95652174]
0.9560274775906844
